# A grid refined at depth, and tracking it in 2D

This notebook builds a MODFLOW 6 DISU model whose upper layers use a coarse Voronoi grid and whose lower layers use finer cells inside a pit. It covers:

1. **`disu.refill`**: re-mesh only the inside of the pit. The cells outside stay the same, so those columns continue through every layer. The new cells end at the same outline vertices, so every outline face is shared one to one.
2. **Gap-free contacts**: where the grid changes between layers, cells above and below meet exactly.
3. **An accuracy check**: a uniform head gradient is reproduced exactly with XT3D, on dipping layers.
4. **Tracking in 2D**: FloPy plots the model layers and cross-sections directly. `idomain_layers`, `layer_view` and `vertex_grid` add per-layer status maps and views of layers that were split into sublayers.

Full connectivity follows Provost, A.M., Bardot, K., Langevin, C.D. and McCallum, J.L. (2025), *Groundwater* 63(3), 399-408, <https://doi.org/10.1111/gwat.13459>.

**Requirements:**

- `python -m pip install -e ".[examples]"`;
- the MODFLOW 6 executable on PATH, or an `MF6_EXE` path (`get-modflow /path/to/bin --subset mf6`).

In [ ]:
import os
import shutil
import tempfile
from pathlib import Path

import numpy as np
from shapely.geometry import LineString, box

from vorflow import ConceptualMesh, MeshGenerator, VoronoiTessellator, disu

# Import the meshing package before FloPy and its plotting backend.
import flopy
import matplotlib.pyplot as plt

mf6_exe = shutil.which(os.environ.get("MF6_EXE", "mf6"))
if mf6_exe is None:
    raise FileNotFoundError("MODFLOW 6 executable not found; set MF6_EXE or add mf6 to PATH.")
workspace = Path(tempfile.mkdtemp(prefix="vorflow_refill_"))

## 1. Base grid and refill

The base grid has 60 m cells. The pit outline is a **barrier line**, so it is made of cell faces. `refill` keeps every cell outside the pit and fills the inside with 12 m cells. Their size grows from the outline spacing (about 28.6 m) towards 12 m with a cell-to-cell ratio of `growth_factor`, as in `add_polygon`; `fields=[...]` takes any size field instead.

In [ ]:
domain = box(0, 0, 1000, 600)
pit = box(400, 200, 600, 400)

cm = ConceptualMesh()
cm.add_polygon(domain, zone_id=1, resolution=60)
cm.add_line(LineString(pit.exterior.coords), line_id="pit", resolution=30, is_barrier=True)
mg = MeshGenerator(background_lc=60, conceptual_mesh=cm, verbosity=0)
mg.generate()
base = VoronoiTessellator(mg, cm).generate()

pit_grid = disu.refill(base, pit, resolution=12, growth_factor=1.2)
print(pit_grid.attrs["refill"])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, g, title in zip(axes, (base, pit_grid), ("base grid", "refilled inside the pit")):
    g.boundary.plot(ax=ax, linewidth=0.5, color="k")
    ax.plot(*pit.exterior.xy, color="tab:red", linewidth=1.5)
    ax.set_title(f"{title} ({len(g)} cells)")
    ax.set_aspect("equal")
# Zoom on one corner: the inside cells end at the base vertices on the outline.
axes[1].set_xlim(380, 480)
axes[1].set_ylim(180, 280)

`border_divisions=2` would split each outline face on the inside, giving smaller cells along the outline while still sharing the base vertices.

## 2. Layers

Five layers dip 3 % to the east. Layers 1-2 use the base grid and layers 3-5 the refilled grid. Inside the pit, layer 4 is split into two sublayers with `disu.subdivide`; outside it stays whole. Layer 2 has a lens that pinches out (idomain -1), and a block of layer 5 is no-flow (idomain 0).

With different grids per layer, the arrays are lists with one entry per layer. `subdivide` works on the layers that share a grid, here the three lower ones.

In [ ]:
xb, xp = base.x.to_numpy(), pit_grid.x.to_numpy()
top = 100.0 - 0.03 * xb
upper_botm = [100.0 - depth - 0.03 * xb for depth in (20.0, 40.0)]
lower_botm = np.array([100.0 - depth - 0.03 * xp for depth in (50.0, 60.0, 80.0)])
inside_pit = (pit_grid.splice_source == "fine").to_numpy()
lower_botm, lower_idomain, lower_parent = disu.subdivide(
    100.0 - 40.0 - 0.03 * xp, lower_botm, nsub=[1, 2, 1], where=[None, inside_pit, None])

grids = [base, base] + [pit_grid] * len(lower_botm)
botm = upper_botm + list(lower_botm)
parent = np.r_[0, 1, 2 + lower_parent]          # original layer of each model layer
lens = (np.hypot(xb - 250, base.y.to_numpy() - 300) < 90)
idomain = [np.ones(len(base), int), np.where(lens, -1, 1)] + [row.copy() for row in lower_idomain]
idomain[-1][xp > 880] = 0

d = disu.DisuGrid(grids, top, botm, idomain).build()
print(d.report())
print("original layer of each model layer:", parent)

The report lists the contact groups at the change of grid. Cells that overlap across it share one elevation, so no gap or overlap is left.

## 3. Accuracy check

A uniform head gradient is a solution MODFLOW should reproduce exactly. Fixed heads go on the outer cells and on the top and bottom layers. Every cell is active here, because no-flow cells would change the solution.

In [ ]:
def run(name, grid, k=1.0, chd=(), wel=(), strt=50.0):
    sim = flopy.mf6.MFSimulation(sim_ws=str(workspace / name), exe_name=mf6_exe)
    flopy.mf6.ModflowTdis(sim)
    flopy.mf6.ModflowIms(sim, outer_dvclose=1e-10, inner_dvclose=1e-11, rcloserecord=1e-9,
                         linear_acceleration="bicgstab", inner_maximum=1000, outer_maximum=100)
    gwf = flopy.mf6.ModflowGwf(sim, modelname="m")
    flopy.mf6.ModflowGwfdisu(gwf, **grid.get_gridprops_disu6())
    # With offset layers, XT3D needs ANGLE2/ANGLE3 (zeros are fine) to use the vertical offsets.
    flopy.mf6.ModflowGwfnpf(gwf, k=k, k33=k, xt3doptions=True, angle2=0.0, angle3=0.0)
    flopy.mf6.ModflowGwfic(gwf, strt=strt)
    flopy.mf6.ModflowGwfchd(gwf, stress_period_data=list(chd))
    if wel:
        flopy.mf6.ModflowGwfwel(gwf, stress_period_data=list(wel))
    flopy.mf6.ModflowGwfoc(gwf, head_filerecord="m.hds", saverecord=[("HEAD", "ALL")])
    sim.write_simulation(silent=True)
    success, output = sim.run_simulation(silent=True, report=True)
    assert success, "\n".join(output[-20:])
    return gwf


check = disu.DisuGrid(grids, top, botm).build()
cm_check = check.cellmap[check.cellmap.node >= 0].sort_values("node")
x, y, layer = cm_check.x.to_numpy(), cm_check.y.to_numpy(), cm_check.layer.to_numpy()
z = ((cm_check.top + cm_check.bot) / 2).to_numpy()
exact = 50.0 - 0.01 * x + 0.005 * y - 0.3 * z
edge = (x < 60) | (x > 940) | (y < 60) | (y > 540) | (layer == 0) | (layer == layer.max())
chd = [[(int(n),), float(exact[n])] for n in np.flatnonzero(edge)]
gwf = run("check", check, chd=chd, strt=float(exact.mean()))
error = np.abs(gwf.output.head().get_data().ravel() - exact)[~edge].max()
print(f"max head error: {error:.2e} m over a {np.ptp(exact):.1f} m head range")

## 4. A well in the pit

Fixed heads go on the west (60 m) and east (50 m) edges, and a well pumps 500 m³/d from the deepest layer in the middle of the pit. `locate` finds the well's node from its coordinates.

In [ ]:
cm = d.cellmap[d.cellmap.node >= 0]
west = cm.node[cm.x < 30].to_numpy()
east = cm.node[cm.x > 970].to_numpy()
chd = [[(int(n),), 60.0] for n in west] + [[(int(n),), 50.0] for n in east]
well = d.locate(500.0, 300.0, 100.0 - 70.0 - 0.03 * 500.0)
print("well node", well, cm.set_index("node").loc[well, ["layer", "x", "y", "top", "bot"]].to_dict())
gwf = run("pit", d, k=5.0, chd=chd, wel=[[(int(well),), -500.0]], strt=55.0)
heads = gwf.output.head().get_data().ravel()
d.save_cellmap(workspace / "cellmap.csv")

### FloPy plots the model layers directly

`get_gridprops_disu6` writes each node's layer number on the IHC diagonal. FloPy reads it, so `gwf.modelgrid` is a layered grid and `PlotMapView(layer=k)` needs no vorflow code.

For a cross-section, FloPy's `PlotCrossSection` misdrew this model: cells in the wrong places and gaps that are not in the grid. The likely cause is the sublayer that exists only inside the pit, though that is not confirmed. The section below is drawn from `cellmap` (each cell's node and elevations) and the plan grids, in a few lines.

In [ ]:
print("cells per model layer (FloPy):", list(gwf.modelgrid.ncpl))
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, k in zip(axes, (1, len(grids) - 1)):
    pmv = flopy.plot.PlotMapView(modelgrid=gwf.modelgrid, layer=k, ax=ax)
    pc = pmv.plot_array(heads, cmap="viridis")
    pmv.plot_grid(linewidth=0.2, color="k")
    plt.colorbar(pc, ax=ax, shrink=0.8)
    ax.set_title(f"model layer {k + 1} heads")
    ax.set_aspect("equal")

from matplotlib.collections import PatchCollection
from matplotlib.patches import Rectangle


def section(ax, d, values, line):
    # Active cells cut by a plan line, drawn at their elevations and coloured by node values.
    patches, colours = [], []
    for layer, grid in enumerate(d.grids_per_layer):
        rows = d.cellmap[d.cellmap.layer == layer].reset_index(drop=True)
        for i, cut in enumerate(grid.geometry.intersection(line)):
            if rows.node[i] < 0 or cut.is_empty or cut.length == 0:
                continue
            x0, x1 = cut.bounds[0], cut.bounds[2]
            patches.append(Rectangle((x0, rows.bot[i]), x1 - x0, rows.top[i] - rows.bot[i]))
            colours.append(values[rows.node[i]])
    collection = PatchCollection(patches, cmap="viridis", edgecolor="k", linewidth=0.3)
    collection.set_array(np.array(colours))
    ax.add_collection(collection)
    ax.autoscale()
    return collection


fig, ax = plt.subplots(figsize=(12, 4))
pc = section(ax, d, heads, LineString([(0, 300), (1000, 300)]))
plt.colorbar(pc, ax=ax, shrink=0.8)
ax.set_title("section along y = 300 m (heads)")

### What vorflow adds: status maps and original-layer views

- **`idomain_layers`**: each layer's cells in MODFLOW 6 codes (1 active, -1 pass-through, 0 no-flow). MODFLOW 6 DISU itself only accepts 0 and 1.
- **`layer_view`**: one value per plan cell for an *original* layer, even after it was split. Pick the top, bottom or middle sublayer, or a thickness-weighted mean.
- **`vertex_grid`**: a FloPy grid of that layer's plan grid, to plot these arrays.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, k in zip(axes, (1, len(grids) - 1)):
    pmv = flopy.plot.PlotMapView(modelgrid=d.vertex_grid(k), ax=ax)
    pc = pmv.plot_array(d.idomain_layers[k], cmap="RdYlGn", vmin=-1, vmax=1)
    pmv.plot_grid(linewidth=0.2, color="k")
    ax.set_title(f"model layer {k + 1}: 1 active, -1 pass-through, 0 no-flow")
    ax.set_aspect("equal")

In [ ]:
# Original layer 4 was split in two inside the pit.
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, pick in zip(axes, ("top", "bottom", "mean")):
    view = d.layer_view(heads, layer=3, parent=parent, pick=pick)
    pmv = flopy.plot.PlotMapView(modelgrid=d.vertex_grid(3), ax=ax)
    pc = pmv.plot_array(view, cmap="viridis")
    plt.colorbar(pc, ax=ax, shrink=0.8)
    ax.set_title(f"original layer 4, pick='{pick}'")
    ax.set_xlim(350, 650)
    ax.set_ylim(150, 450)
    ax.set_aspect("equal")

In [ ]:
# The pinched lens in layer 2: empty by default, or shown with the cell that took its interval.
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, fill in zip(axes, ("nan", "cover")):
    view = d.layer_view(heads, layer=1, parent=parent, fill=fill)
    pmv = flopy.plot.PlotMapView(modelgrid=d.vertex_grid(1), ax=ax)
    pc = pmv.plot_array(view, cmap="viridis")
    plt.colorbar(pc, ax=ax, shrink=0.8)
    ax.set_title(f"layer 2, fill='{fill}'")
    ax.set_aspect("equal")

**Notes:**

- **`cellmap`** (saved to CSV above with `save_cellmap`) lists every cell with its node (-1 when removed), layer, grid, `node_id`, status and elevations.
- **Merging small pieces** (`sliver_abs_tol`) is off by default. Merging gives fewer connections, but moves face area between cells, so the results are no longer exact.
- **`splice`** joins two separately made grids. Their outline vertices need not match (hanging vertices are fine for DISU). `refill` is the way to share them.